# 04 SmolVLA 在 ROCm 上的迁移与采样加权

        这一节用 SmolVLA 说明：语言策略不能只看总体成功率。大家会把红杯和蓝杯指令拆开评估，再比较复制 episode 与 Weighted sampler 的效果。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：读取红杯/蓝杯固定指令结果


In [ ]:
snapshot = json.loads((ASSET_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
rows = []
for item in snapshot["smolvla_forced_instruction_n10"]:
    rows.append((item["label"], item["red_text"], item["blue_text"]))
md_table(["策略", "红杯 physical_success", "蓝杯 physical_success"], rows)
show_asset("smolvla_red_blue_success.png", width=1100)


baseline 红杯好、蓝杯差，说明模型不是完全不会抓，而是任务条件和颜色相关分布没有学稳。


## Checkpoint 2：复制 episode 与 Weighted sampler 的区别


In [ ]:
rows = [
    ("复制 episode", "直接改变数据集 episode 分布", "实现简单，但容易让模型偏向某个颜色"),
    ("Weighted sampler", "不改原始 parquet，只改采样概率", "更容易回滚，也更适合小数据平衡"),
    ("中间 checkpoint 评估", "不要只看 final step", "本示例里 step500 比 step1000 更均衡"),
]
md_table(["方法", "做法", "为什么要注意"], rows)


## Checkpoint 3：重新生成图表


In [ ]:
cmd = [
    sys.executable,
    str(TOPIC_ROOT / "code" / "generate_tutorial_assets.py"),
    "--source-root",
    str(OUTPUT_ROOT),
]
print("如果 OUTPUT_ROOT 中有本章需要的 TSV/JSONL/MP4，可以运行：")
print(" ".join(cmd))
# subprocess.run(cmd, check=True)


上面的单元默认不直接执行，避免大家还没准备好输出目录时生成占位图。确认 `$OUTPUT_ROOT` 正确后，取消最后一行注释即可。


## Checkpoint 4：蓝杯失败和修复后成功对照


In [ ]:
show_asset("smolvla_blue_failure_sequence.jpg", width=1100)
show_asset("smolvla_blue_success_sequence.jpg", width=1100)
